In [31]:
import jax
import jax.numpy as jnp

import jrystal

In [2]:
jax.config.update("jax_debug_nans", True)

## Create a crystal object

In [3]:
diamond_file_path = "../geometry/diamond.xyz"
diamond = jrystal.Crystal.create_from_file(diamond_file_path)

In [4]:
diamond.vol

Array(76.54842, dtype=float32)

In [5]:
diamond.num_electron

Array(12, dtype=int32)

In [6]:
num_bands = diamond.num_electron

## Grids

In [7]:
kpts = jrystal.grid.k_vectors(diamond.cell_vectors, grid_sizes = [2, 2, 2])
g_vecs = jrystal.grid.g_vectors(diamond.cell_vectors, grid_sizes = [24, 24, 24])

In [8]:
num_kpts = kpts.shape[0]
num_kpts

8

In [9]:
freq_mask = jrystal.grid.spherical_mask(diamond.cell_vectors, grid_sizes = [24, 24, 24], cutoff_energy=40)
freq_mask.shape

(24, 24, 24)

## Occupation

In [10]:
key = jax.random.PRNGKey(123)
params_occ = jrystal.occupation.idempotent_param_init(key, num_bands, num_kpts)
occupation = jrystal.occupation.idempotent(params_occ, num_electrons=diamond.num_electron, num_kpts=num_kpts, spin=0, polarize=False)
occupation.shape

(1, 8, 12)

In [11]:
def occ(params):
    return jnp.sum(jrystal.occupation.idempotent(params, num_electrons=diamond.num_electron, num_kpts=num_kpts, spin=0, polarize=False))

jax.grad(occ)(params_occ)

{'w_im': Array(0., dtype=float32, weak_type=True),
 'w_re': Array([[-1.0602322e-05,  2.1693361e-06, -8.3233635e-06, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00],
        [-1.4879618e-06, -1.9298623e-06, -7.1519007e-06, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00],
        [-1.0544281e-06, -6.6345374e-06,  6.8078566e-06, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00],
        ...,
        [ 8.9687965e-06, -4.4475187e-06, -4.7997173e-06, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00],
        [ 1.0428734e-05,  1.9524480e-06, -5.5149603e-06, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00],
        [ 1.1439815e-05, -1.3080557e-06, -1.1125037e-05, ...,
         -0.0000000e+00,  0.0000000e+00, -0.0000000e+00]], dtype=float32)}

## Planewave

In [12]:
key = jax.random.PRNGKey(123)
params = jrystal.pw.pw_param_init(key, num_bands=diamond.num_electron, num_kpts=kpts.shape[0], freq_mask=freq_mask, polarize=False)
coeff = jrystal.pw.pw_coeff(params, freq_mask)
coeff.shape

(1, 8, 12, 24, 24, 24)

In [13]:
# density
density = jrystal.pw.density_grid(coeff, diamond.vol, occupation)
density.shape

(24, 24, 24)

## total energy

In [14]:
total_energy = jrystal.energy.total_energy(
    coeff, positions=diamond.positions, charges=diamond.charges, 
    g_vector_grid=g_vecs, kpts=kpts, vol=diamond.vol, 
    occupation=occupation, kohn_sham=False, split=False
)
total_energy

Array(288.53445, dtype=float32)

In [15]:
density_grid_reciprocal = jrystal.pw.density_grid_reciprocal(coeff, diamond.vol, occupation)
hartree = jrystal.energy.hartree(density_grid_reciprocal, g_vecs, diamond.vol, False)

In [16]:
hartree

Array(0.9843851, dtype=float32)

In [17]:
def g(positions):
    return jrystal.energy.total_energy(
    coeff, positions=positions, charges=diamond.charges, 
    g_vector_grid=g_vecs, kpts=kpts, vol=diamond.vol, 
    occupation=occupation, kohn_sham=False, split=False
    )

forces = jax.grad(g)(1.0 * diamond.positions)

In [19]:
def h(charges):
    return jrystal.energy.total_energy(
    coeff, positions=diamond.positions, charges=charges, 
    g_vector_grid=g_vecs, kpts=kpts, vol=diamond.vol, 
    occupation=occupation, kohn_sham=False, split=False
    )

jax.grad(h)(1.0 * diamond.charges)

Array([-0.07491288, -0.07162403], dtype=float32)

In [33]:
def m(geo):
    pos, cha = geo
    hamiltonian = jrystal.hamiltonian.hamiltonian_matrix(coeff, pos, cha, g_vecs, kpts, diamond.vol, occupation, kohn_sham=True)
    eigs = jnp.linalg.eigvalsh(hamiltonian)
    return eigs

def loss(geo):
    eigs = m(geo)
    return jnp.mean((500-eigs) ** 2)

geo = (diamond.positions, diamond.charges * 1.0)
jax.grad(loss)(geo)

(Array([[-3027.3416, -3032.3928, -2989.6453],
        [ 3083.6792,  3021.911 ,  3092.8987]], dtype=float32),
 Array([-161.83618, -163.14581], dtype=float32))